### Import the necessary libraries

In [ ]:
import os
from zipfile import ZipFile
from ultralytics import YOLO
import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import KFold, train_test_split
from collections import defaultdict
import glob

Set path

In [ ]:
path = os.getcwd()
print(path)

The data should be in a zip file.

Here either Dataset A or B can be used.

In [ ]:
# Create the directory if it doesn't exist
extract_dir = "data"
os.makedirs(extract_dir, exist_ok=True)

#adjust filename if necessary
with ZipFile(path + "/glass-defect-detection-v3i-yolov11-2.zip", "r") as data_set:
    data_set.extractall(extract_dir)

(Down)load model

In [ ]:
try:
    model = YOLO("yolo11n.pt")
except:
    model = YOLO("yolo11n.pt")

### PRE-WORK

Initialize workflow variables

In [ ]:
N_SPLITS = 5  # for CV
MODEL_SIZE = "yolo11n.pt"
EPOCHS_PER_FOLD = 100
TEST_SIZE = 0.2

# Base paths
BASE_DATA_PATH = path + "/data/Glass Defect Detection.v3i.yolov11" # i.e /home/user/yolo_data/
TEMP_DIR = os.path.abspath("./cv_temp/") # Temporäre Dateien werden hier gespeichert

# YOLO-Classnames
CLASS_NAMES = ["defect", "glass"]
NC = len(CLASS_NAMES)
# --------------------------------------------------------

# Folder for temp files
os.makedirs(TEMP_DIR, exist_ok=True)
all_fold_metrics = defaultdict(list)
results_df = None

In [ ]:
BASE_DATA_PATH

Load the images

In [ ]:
IMAGE_FOLDER = os.path.join(BASE_DATA_PATH, "images")

# Lists all images and saves them to X
X = np.array(sorted(glob.glob(os.path.join(IMAGE_FOLDER, "*.jpg"))))

total_samples = len(X)
print(f"Loaded Images: {total_samples}")

# Dummy-y for KFold
y = np.zeros(total_samples)

Split the data into 80% Train and 20% Test

In [ ]:
X_pool, X_test, _, _ = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=42,
    shuffle=True
)

print(f"Total data: {len(X)}")
print(f"Test-Set (20%): {len(X_test)} Samples. Will be ignored by CV.")
print(f"Training-Pool (80%): {len(X_pool)} Samples.")

### Training Logic

In [ ]:
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# First generate the constant YAML for the Test set.
test_file_path = os.path.join(TEMP_DIR, 'final_test.txt')
np.savetxt(test_file_path, X_test, fmt='%s')

best_mAP50_95 = 0.0
best_model_path = ""
best_fold_number = 0

for fold, (train_index, val_index) in enumerate(kf.split(X_pool)): # Split only on X_pool!
    print(f"\n==================================================")
    print(f"--- FOLD {fold+1}/{N_SPLITS} starting... ---")
    print(f"==================================================")

    # 1. Split of train pool into train and val
    X_train = X_pool[train_index]
    X_val = X_pool[val_index]

    # 2. Generation of temporary index files.
    train_file_path = os.path.join(TEMP_DIR, f'fold_{fold}_train.txt')
    val_file_path = os.path.join(TEMP_DIR, f'fold_{fold}_val.txt')

    np.savetxt(train_file_path, X_train, fmt='%s')
    np.savetxt(val_file_path, X_val, fmt='%s')

    # 3. temporary yolo configuration YAML.
    temp_yaml_data = {
        'path': BASE_DATA_PATH,
        'train': train_file_path,
        'val': val_file_path,
        'nc': NC,
        'names': CLASS_NAMES
    }
    temp_yaml_path = os.path.join(TEMP_DIR, f'fold_{fold}.yaml')
    with open(temp_yaml_path, 'w') as f:
        yaml.dump(temp_yaml_data, f)

    # 4. New initialisation and training of model
    model = YOLO(MODEL_SIZE)
    run_name = f'yolo_cv_fold_{fold+1}'
    project_dir = './yolo_runs'

    results = model.train(
        data=temp_yaml_path,
        epochs=EPOCHS_PER_FOLD,
        name=run_name,
        project=project_dir,
        val=True,
        lr0=0.0001
    )

    # 5. Extract metrics and save best weight
    final_metrics = results.results_dict
    current_mAP50_95 = final_metrics.get('metrics/mAP50-95(B)', 0.0)

    # Saving of all metrics
    all_fold_metrics['precision'].append(final_metrics.get('metrics/precision(B)', 0.0))
    all_fold_metrics['recall'].append(final_metrics.get('metrics/recall(B)', 0.0))
    all_fold_metrics['mAP50'].append(final_metrics.get('metrics/mAP50(B)', 0.0))
    all_fold_metrics['mAP50-95'].append(current_mAP50_95)
    all_fold_metrics['fitness'].append(final_metrics.get('fitness', 0.0))
    all_fold_metrics['fold'].append(fold + 1)

    # Follow best model
    if current_mAP50_95 > best_mAP50_95:
        best_mAP50_95 = current_mAP50_95
        best_fold_number = fold + 1
        best_model_path = os.path.join(project_dir, run_name, 'weights', 'best.pt')
        print(f"-> NEUES BESTES MODELL gefunden in Fold {best_fold_number} mit mAP50-95: {best_mAP50_95:.4f}")


    # 6. Cleaning
    os.remove(train_file_path)
    os.remove(val_file_path)
    os.remove(temp_yaml_path)

### Results of CV

In [ ]:
# Convert Data into DF
if all_fold_metrics['fold']:
    results_df = pd.DataFrame(all_fold_metrics)

    # Calc of mean and std.
    summary_metrics = results_df.drop(columns=['fold']).agg(['mean', 'std'])

    print("\n\n##################################################")
    print("## ENDRESULT OF 5FOLD-CV ##")
    print("##################################################")

    print("\n--- Metrics per Fold ---")
    print(results_df)

    print("\n--- Summary (Mean ± Std.) ---")
    for metric in summary_metrics.columns:
        mean = summary_metrics.loc['mean', metric]
        std = summary_metrics.loc['std', metric]
        print(f"  {metric}: Mean = {mean:.4f} (± {std:.4f} Std.)")
else:
    print("ERROR: No metrics collected.")

### Testing

In [ ]:
print("\n##################################################")
print("## FINAL EVALUATION ON 20% TEST-SET ##")
print("##################################################")

# Creation of temporary yaml for testing
final_test_yaml_path = os.path.join(TEMP_DIR, 'final_test_config.yaml')

# 1. Writing final test set into a txt.
test_file_path = os.path.join(TEMP_DIR, 'final_test_paths.txt')
np.savetxt(test_file_path, X_test, fmt='%s') # X_test contains 20%


final_test_yaml_data = {
    'path': BASE_DATA_PATH, # Basis-Pfad zu Bildern/Labels
    'train': os.path.join(TEMP_DIR, 'dummy_train.txt'), # Dummy-Pfad
    'val': os.path.join(TEMP_DIR, 'dummy_val.txt'),     # Dummy-Pfad
    'test': test_file_path,                             # Pfadliste für den Test-Split
    'nc': NC,
    'names': CLASS_NAMES
}

# Dummy-Data because the model needs val and train to work
with open(os.path.join(TEMP_DIR, 'dummy_train.txt'), 'w') as f: f.write('dummy')
with open(os.path.join(TEMP_DIR, 'dummy_val.txt'), 'w') as f: f.write('dummy')

with open(final_test_yaml_path, 'w') as f:
    yaml.dump(final_test_yaml_data, f)

if best_model_path:
    # load best model from CV
    final_model = YOLO(best_model_path)
    
    print(f"Starting final evaluation for best weight ({os.path.basename(best_model_path)}) on 20%-Test-Set...")

    # 3. Start test
    metrics = final_model.val(
        data=final_test_yaml_path,
        split="test",
        batch=16
    )
    
    # 4. Print metrics
    final_mAP = metrics.box.map 
    final_mAP50 = metrics.box.map50
    final_precision = metrics.box.p
    final_recall = metrics.box.r

    print(f"\nFinal result of best weight from CV ({best_fold_number}) on 20% Test-Set:")
    print(f"  mAP (mAP50-95): {final_mAP:}")
    print(f"  mAP50:          {final_mAP50:}")
    print(f"  Precision (P):  {final_precision:}")
    print(f"  Recall (R):     {final_recall:}")
    
    # Aufräumen
    os.remove(test_file_path)
    os.remove(final_test_yaml_path)
    os.remove(os.path.join(TEMP_DIR, 'dummy_train.txt'))
    os.remove(os.path.join(TEMP_DIR, 'dummy_val.txt'))
else:
    print("No best model found, check CV output.")